In [13]:
import os
folders = [
    'data/raw',
    'data/processed',
    'notebooks',
    'sql',
    'dashboard',
    'reports'
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)
    print(f" Created — {folder}")



 Created — data/raw
 Created — data/processed
 Created — notebooks
 Created — sql
 Created — dashboard
 Created — reports


In [14]:
pip install pandas numpy matplotlib seaborn plotly sqlalchemy requests scipy jupyter

Note: you may need to restart the kernel to use updated packages.


In [15]:
requirements = """pandas
numpy
matplotlib
seaborn
plotly
sqlalchemy
requests
scipy
jupyter
"""

with open('requirements.txt', 'w') as f:
    f.write(requirements)

print(requirements)

pandas
numpy
matplotlib
seaborn
plotly
sqlalchemy
requests
scipy
jupyter



In [17]:
import os

print("Current folder:")
print(os.getcwd())
print()
print("All files here:")
for f in os.listdir():
    print(f"  {'📁' if os.path.isdir(f) else '📄'} {f}")

Current folder:
C:\Users\MERLIN J

All files here:
  📁 .anaconda
  📁 .cache
  📁 .codex
  📁 .conda
  📁 .continuum
  📄 .gitconfig
  📁 .idlerc
  📁 .ipynb_checkpoints
  📁 .ipython
  📁 .jupyter
  📄 .lesshst
  📁 .m2
  📁 .matplotlib
  📁 .ms-ad
  📄 .python_history
  📁 .rustup
  📁 .streamlit
  📁 .th-client
  📄 .viminfo
  📁 .VirtualBox
  📁 .vscode
  📁 .vscode-shared
  📄 01_fund_master.csv
  📄 02_nav_history.csv
  📄 03_aum_by_fund_house.csv
  📄 04_monthly_sip_inflows.csv
  📄 05_category_inflows.csv
  📄 06_industry_folio_count.csv
  📄 07_scheme_performance.csv
  📄 08_investor_transactions.csv
  📄 09_portfolio_holdings.csv
  📄 10_benchmark_indices.csv
  📁 ai-resume-builder
  📄 ai_dashboard.html
  📁 anaconda3
  📁 anaconda_projects
  📄 analytics_report.pdf
  📁 AppData
  📁 Application Data
  📄 business_insights.txt
  📄 cleaned_dataset.csv
  📁 Contacts
  📁 Cookies
  📄 Customer_Churn_Analysis.ipynb
  📄 customer_dna.csv
  📄 customer_dna_cards.png
  📄 customer_shopping_behavior.csv
  📁 dashboard
  📁 data


In [18]:
import pandas as pd
import os

RAW_DIR = ""  # current folder

DATASETS = {
    "fund_master"         : "01_fund_master.csv",
    "nav_history"         : "02_nav_history.csv",
    "aum_by_fund_house"   : "03_aum_by_fund_house.csv",
    "monthly_sip_inflows" : "04_monthly_sip_inflows.csv",
    "category_inflows"    : "05_category_inflows.csv",
    "industry_folio_count": "06_industry_folio_count.csv",
    "scheme_performance"  : "07_scheme_performance.csv",
    "investor_transactions": "08_investor_transactions.csv",
    "portfolio_holdings"  : "09_portfolio_holdings.csv",
    "benchmark_indices"   : "10_benchmark_indices.csv",
}

dataframes = {}
anomalies  = []

print("=" * 60)
print("   LOADING ALL 10 CSV DATASETS")
print("=" * 60)

for name, path in DATASETS.items():
    if os.path.exists(path):
        df = pd.read_csv(path)
        dataframes[name] = df

        print(f"\n {name.upper()}")
        print(f"   Shape   : {df.shape}")
        print(f"   Columns : {df.columns.tolist()}")
        print(f"\n   Data Types:")
        print(df.dtypes.to_string())
        print(f"\n   First 3 Rows:")
        print(df.head(3).to_string())

        # Check anomalies
        missing = df.isnull().sum().sum()
        dupes   = df.duplicated().sum()
        if missing > 0:
            anomalies.append(
                f"{name}: {missing} missing values")
        if dupes > 0:
            anomalies.append(
                f"{name}: {dupes} duplicate rows")

        print(f"\n   Missing Values : {missing}")
        print(f"   Duplicates     : {dupes}")
        print("-" * 60)
    else:
        print(f"\n {name} — NOT FOUND!")
        anomalies.append(f"{name}: File not found!")

# ── Copy to data/raw folder ────────────────────────────
print("\nCopying files to data/raw folder...")
import shutil
os.makedirs('data/raw', exist_ok=True)
for name, path in DATASETS.items():
    if os.path.exists(path):
        shutil.copy(path, f"data/raw/{path}")
        print(f"   Copied — {path}")

print(f"\n{'='*60}")
print("ANOMALIES FOUND:")
if anomalies:
    for a in anomalies:
        print(f"    {a}")
else:
    print("   No anomalies found!")
print(f"{'='*60}")
print(f"\n Total datasets loaded: {len(dataframes)}")

   LOADING ALL 10 CSV DATASETS

 FUND_MASTER
   Shape   : (40, 15)
   Columns : ['amfi_code', 'fund_house', 'scheme_name', 'category', 'sub_category', 'plan', 'launch_date', 'benchmark', 'expense_ratio_pct', 'exit_load_pct', 'min_sip_amount', 'min_lumpsum_amount', 'fund_manager', 'risk_category', 'sebi_category_code']

   Data Types:
amfi_code               int64
fund_house             object
scheme_name            object
category               object
sub_category           object
plan                   object
launch_date            object
benchmark              object
expense_ratio_pct     float64
exit_load_pct         float64
min_sip_amount          int64
min_lumpsum_amount      int64
fund_manager           object
risk_category          object
sebi_category_code     object

   First 3 Rows:
   amfi_code       fund_house                                 scheme_name category sub_category     plan launch_date             benchmark  expense_ratio_pct  exit_load_pct  min_sip_amount  min_lu

In [19]:
import requests
import pandas as pd
import json

print("=" * 60)
print("   FETCHING LIVE NAV — HDFC TOP 100 DIRECT")
print("=" * 60)

url      = "https://api.mfapi.in/mf/125497"
response = requests.get(url)

if response.status_code == 200:
    data = response.json()

    # ── Parse response ─────────────────────────────────
    scheme_name = data['meta']['scheme_name']
    fund_house  = data['meta']['fund_house']
    scheme_type = data['meta']['scheme_type']
    nav_data    = data['data']

    print(f"\n  Scheme Name : {scheme_name}")
    print(f"  Fund House  : {fund_house}")
    print(f"  Scheme Type : {scheme_type}")
    print(f"  NAV Records : {len(nav_data)}")
    print(f"\n  Latest NAV  : {nav_data[0]['nav']}")
    print(f"  Latest Date : {nav_data[0]['date']}")

    # ── Save as CSV ────────────────────────────────────
    df_nav = pd.DataFrame(nav_data)
    df_nav['scheme_name'] = scheme_name
    df_nav['fund_house']  = fund_house
    df_nav.to_csv('data/raw/hdfc_top100_nav.csv', index=False)

    print(f"\n Saved as data/raw/hdfc_top100_nav.csv")
    print(df_nav.head())
else:
    print(f" API Error: {response.status_code}")

   FETCHING LIVE NAV — HDFC TOP 100 DIRECT

  Scheme Name : SBI Small Cap Fund - Direct Plan - Growth
  Fund House  : SBI Mutual Fund
  Scheme Type : Open Ended Schemes
  NAV Records : 3109

  Latest NAV  : 203.91570
  Latest Date : 25-06-2026

 Saved as data/raw/hdfc_top100_nav.csv
         date        nav                                scheme_name       fund_house
0  25-06-2026  203.91570  SBI Small Cap Fund - Direct Plan - Growth  SBI Mutual Fund
1  24-06-2026  205.06550  SBI Small Cap Fund - Direct Plan - Growth  SBI Mutual Fund
2  23-06-2026  203.54430  SBI Small Cap Fund - Direct Plan - Growth  SBI Mutual Fund
3  22-06-2026  204.12630  SBI Small Cap Fund - Direct Plan - Growth  SBI Mutual Fund
4  19-06-2026  202.07610  SBI Small Cap Fund - Direct Plan - Growth  SBI Mutual Fund


In [20]:
import requests
import pandas as pd
import time

schemes = {
    "SBI Bluechip"       : 119551,
    "ICICI Bluechip"     : 120503,
    "Nippon Large Cap"   : 118632,
    "Axis Bluechip"      : 119092,
    "Kotak Bluechip"     : 120841,
}

print("=" * 60)
print("   FETCHING NAV FOR 5 KEY SCHEMES")
print("=" * 60)

all_nav = []

for scheme_name, code in schemes.items():
    url      = f"https://api.mfapi.in/mf/{code}"
    response = requests.get(url)

    if response.status_code == 200:
        data     = response.json()
        nav_data = data['data']
        meta     = data['meta']

        latest_nav  = nav_data[0]['nav']
        latest_date = nav_data[0]['date']

        print(f"\n   {scheme_name}")
        print(f"     Code       : {code}")
        print(f"     Full Name  : {meta['scheme_name']}")
        print(f"     Latest NAV : Rs.{latest_nav}")
        print(f"     Date       : {latest_date}")
        print(f"     Records    : {len(nav_data)}")

        # Add to combined list
        for record in nav_data:
            all_nav.append({
                'scheme_code' : code,
                'scheme_name' : scheme_name,
                'full_name'   : meta['scheme_name'],
                'fund_house'  : meta['fund_house'],
                'date'        : record['date'],
                'nav'         : record['nav']
            })

        # Save individual file
        df_scheme = pd.DataFrame(nav_data)
        df_scheme['scheme_code'] = code
        df_scheme['scheme_name'] = scheme_name
        filename = f"data/raw/{scheme_name.replace(' ','_').lower()}_nav.csv"
        df_scheme.to_csv(filename, index=False)
        print(f"     Saved      : {filename}")

    else:
        print(f"\n   {scheme_name} — Error {response.status_code}")

    time.sleep(0.5)  # Be nice to API

# Save combined file
df_all = pd.DataFrame(all_nav)
df_all.to_csv('data/raw/all_5_schemes_nav.csv', index=False)

print(f"\n{'='*60}")
print(f" Combined file saved: data/raw/all_5_schemes_nav.csv")
print(f"   Total Records : {len(df_all)}")
print(f"{'='*60}")
print(df_all.head(10))

   FETCHING NAV FOR 5 KEY SCHEMES

   SBI Bluechip
     Code       : 119551
     Full Name  : Aditya Birla Sun Life Banking & PSU Debt Fund  - DIRECT - IDCW
     Latest NAV : Rs.106.30570
     Date       : 25-06-2026
     Records    : 3254
     Saved      : data/raw/sbi_bluechip_nav.csv

   ICICI Bluechip
     Code       : 120503
     Full Name  : Axis ELSS Tax Saver Fund - Direct Plan - Growth Option
     Latest NAV : Rs.108.37580
     Date       : 25-06-2026
     Records    : 3325
     Saved      : data/raw/icici_bluechip_nav.csv

   Nippon Large Cap
     Code       : 118632
     Full Name  : Nippon India Large Cap Fund - Direct Plan Growth Plan - Growth Option
     Latest NAV : Rs.101.19170
     Date       : 25-06-2026
     Records    : 3316
     Saved      : data/raw/nippon_large_cap_nav.csv

   Axis Bluechip
     Code       : 119092
     Full Name  : HDFC Money Market Fund - Growth Option - Direct Plan
     Latest NAV : Rs.6209.19070
     Date       : 25-06-2026
     Records    : 

In [22]:
import pandas as pd

df_fm = pd.read_csv('01_fund_master.csv')

print("Actual Column Names:")
for col in df_fm.columns:
    print(f"  → {col}")

print(f"\nFirst 3 rows:")
print(df_fm.head(3).to_string())

Actual Column Names:
  → amfi_code
  → fund_house
  → scheme_name
  → category
  → sub_category
  → plan
  → launch_date
  → benchmark
  → expense_ratio_pct
  → exit_load_pct
  → min_sip_amount
  → min_lumpsum_amount
  → fund_manager
  → risk_category
  → sebi_category_code

First 3 rows:
   amfi_code       fund_house                                 scheme_name category sub_category     plan launch_date             benchmark  expense_ratio_pct  exit_load_pct  min_sip_amount  min_lumpsum_amount   fund_manager risk_category sebi_category_code
0     119551  SBI Mutual Fund   SBI Bluechip Fund - Regular Plan - Growth   Equity    Large Cap  Regular  2006-02-14         NIFTY 100 TRI             1.5400         1.0000             500                1000  Sohini Andani      Moderate               EC01
1     119552  SBI Mutual Fund    SBI Bluechip Fund - Direct Plan - Growth   Equity    Large Cap   Direct  2013-01-01         NIFTY 100 TRI             0.6600         1.0000             500        

In [24]:
import pandas as pd

df_fm = pd.read_csv('01_fund_master.csv')

print("=" * 60)
print("   FUND MASTER EXPLORATION")
print("=" * 60)

print(f"\n  Total Schemes    : {len(df_fm)}")
print(f"  Total Columns    : {len(df_fm.columns)}")

print(f"\n  Unique Fund Houses ({df_fm['fund_house'].nunique()}):")
for fh in df_fm['fund_house'].unique():
    print(f"    → {fh}")

print(f"\n  Unique Categories ({df_fm['category'].nunique()}):")
for cat in df_fm['category'].unique():
    print(f"    → {cat}")

print(f"\n  Unique Sub-Categories ({df_fm['sub_category'].nunique()}):")
for sub in df_fm['sub_category'].unique():
    print(f"    → {sub}")

print(f"\n  Unique Risk Categories ({df_fm['risk_category'].nunique()}):")
for risk in df_fm['risk_category'].unique():
    print(f"    → {risk}")

print(f"\n  Unique Plans ({df_fm['plan'].nunique()}):")
for plan in df_fm['plan'].unique():
    print(f"    → {plan}")

print(f"\n  AMFI Scheme Code Structure:")
print(f"    Min Code : {df_fm['amfi_code'].min()}")
print(f"    Max Code : {df_fm['amfi_code'].max()}")
print(f"    Sample   : {df_fm['amfi_code'].head(5).tolist()}")

print(f"\n  Top 5 Fund Houses by Scheme Count:")
print(df_fm['fund_house'].value_counts().head(5).to_string())

print(f"\n  Category Distribution:")
print(df_fm['category'].value_counts().to_string())

print(f"\n  Risk Category Distribution:")
print(df_fm['risk_category'].value_counts().to_string())

print(f"\n  Expense Ratio Stats:")
print(df_fm['expense_ratio_pct'].describe().round(4).to_string())

print(f"\n Fund Master Exploration Complete")

   FUND MASTER EXPLORATION

  Total Schemes    : 40
  Total Columns    : 15

  Unique Fund Houses (10):
    → SBI Mutual Fund
    → HDFC Mutual Fund
    → ICICI Prudential MF
    → Nippon India MF
    → Kotak Mahindra MF
    → Axis Mutual Fund
    → Aditya Birla Sun Life MF
    → UTI Mutual Fund
    → Mirae Asset MF
    → DSP Mutual Fund

  Unique Categories (2):
    → Equity
    → Debt

  Unique Sub-Categories (12):
    → Large Cap
    → Small Cap
    → Gilt
    → Mid Cap
    → Short Duration
    → Value
    → Liquid
    → Index/ETF
    → Flexi Cap
    → Index
    → Large & Mid Cap
    → ELSS

  Unique Risk Categories (5):
    → Moderate
    → Very High
    → Low
    → High
    → Moderately High

  Unique Plans (2):
    → Regular
    → Direct

  AMFI Scheme Code Structure:
    Min Code : 100016
    Max Code : 149324
    Sample   : [119551, 119552, 119598, 119599, 119120]

  Top 5 Fund Houses by Scheme Count:
fund_house
SBI Mutual Fund        5
HDFC Mutual Fund       5
ICICI Prudential

In [27]:
import pandas as pd
import os

df_fm  = pd.read_csv('01_fund_master.csv')
df_nav = pd.read_csv('02_nav_history.csv')

print("=" * 60)
print("   AMFI CODE VALIDATION")
print("=" * 60)

# ── Fixed: amfi_code not scheme_code ──────────────────
fm_codes  = set(df_fm['amfi_code'].unique())
nav_codes = set(df_nav['amfi_code'].unique())

# Find mismatches
missing_in_nav = fm_codes - nav_codes
missing_in_fm  = nav_codes - fm_codes
matched        = fm_codes & nav_codes

print(f"\n  Fund Master Codes  : {len(fm_codes)}")
print(f"  NAV History Codes  : {len(nav_codes)}")
print(f"  Matching Codes     : {len(matched)}")
print(f"  Missing in NAV     : {len(missing_in_nav)}")
print(f"  Extra in NAV       : {len(missing_in_fm)}")

if missing_in_nav:
    print(f"\n   Codes in FM not in NAV:")
    for code in list(missing_in_nav)[:10]:
        print(f"    → {code}")

if missing_in_fm:
    print(f"\n    Codes in NAV not in FM:")
    for code in list(missing_in_fm)[:10]:
        print(f"    → {code}")

# ── Data Quality Summary ───────────────────────────────
print(f"\n{'='*60}")
print("   DATA QUALITY SUMMARY")
print(f"{'='*60}")

summary = (
    "DATA QUALITY SUMMARY — Mutual Fund Analysis\n"
    "\n"
    "1. Fund Master Dataset\n"
    f"   Total Schemes    : {len(df_fm)}\n"
    f"   Total Columns    : {len(df_fm.columns)}\n"
    f"   Missing Values   : {df_fm.isnull().sum().sum()}\n"
    f"   Duplicate Rows   : {df_fm.duplicated().sum()}\n"
    f"   Unique AMFI Codes: {len(fm_codes)}\n"
    f"   Categories       : {df_fm['category'].nunique()}\n"
    f"   Fund Houses      : {df_fm['fund_house'].nunique()}\n"
    "\n"
    "2. NAV History Dataset\n"
    f"   Total Records    : {len(df_nav)}\n"
    f"   Total Columns    : {len(df_nav.columns)}\n"
    f"   Missing Values   : {df_nav.isnull().sum().sum()}\n"
    f"   Duplicate Rows   : {df_nav.duplicated().sum()}\n"
    f"   Unique Schemes   : {len(nav_codes)}\n"
    f"   Date Range       : {df_nav['date'].min()} to {df_nav['date'].max()}\n"
    "\n"
    "3. AMFI Code Validation\n"
    f"   FM Codes         : {len(fm_codes)}\n"
    f"   NAV Codes        : {len(nav_codes)}\n"
    f"   Matched Codes    : {len(matched)}\n"
    f"   Missing in NAV   : {len(missing_in_nav)}\n"
    f"   Extra in NAV     : {len(missing_in_fm)}\n"
    f"   Status           : "
    f"{'All codes matched!' if len(missing_in_nav) == 0 else 'Some codes missing!'}\n"
    "\n"
    "4. Overall Assessment\n"
    f"   Fund Master      : "
    f"{'Clean' if df_fm.isnull().sum().sum() == 0 else 'Has Issues'}\n"
    f"   NAV History      : "
    f"{'Clean' if df_nav.isnull().sum().sum() == 0 else 'Has Issues'}\n"
    f"   Code Consistency : "
    f"{'Consistent' if len(missing_in_nav) == 0 else 'Inconsistent'}\n"
)

print(summary)

# Save summary
os.makedirs('reports', exist_ok=True)
with open('reports/data_quality_summary.txt', 'w') as f:
    f.write(summary)



   AMFI CODE VALIDATION

  Fund Master Codes  : 40
  NAV History Codes  : 40
  Matching Codes     : 40
  Missing in NAV     : 0
  Extra in NAV       : 0

   DATA QUALITY SUMMARY
DATA QUALITY SUMMARY — Mutual Fund Analysis

1. Fund Master Dataset
   Total Schemes    : 40
   Total Columns    : 15
   Missing Values   : 0
   Duplicate Rows   : 0
   Unique AMFI Codes: 40
   Categories       : 2
   Fund Houses      : 10

2. NAV History Dataset
   Total Records    : 46000
   Total Columns    : 3
   Missing Values   : 0
   Duplicate Rows   : 0
   Unique Schemes   : 40
   Date Range       : 2022-01-03 to 2026-05-29

3. AMFI Code Validation
   FM Codes         : 40
   NAV Codes        : 40
   Matched Codes    : 40
   Missing in NAV   : 0
   Extra in NAV     : 0
   Status           : All codes matched!

4. Overall Assessment
   Fund Master      : Clean
   NAV History      : Clean
   Code Consistency : Consistent



In [28]:
code = '''import os
import pandas as pd

# Create folders
folders = [
    "data/raw", "data/processed",
    "notebooks", "sql",
    "dashboard", "reports"
]
for folder in folders:
    os.makedirs(folder, exist_ok=True)

RAW_DIR = "data/raw"

DATASETS = {
    "fund_master"        : f"{RAW_DIR}/01_fund_master.csv",
    "nav_history"        : f"{RAW_DIR}/02_nav_history.csv",
    "aum_by_fund_house"  : f"{RAW_DIR}/03_aum_by_fund_house.csv",
    "scheme_returns"     : f"{RAW_DIR}/04_scheme_returns.csv",
    "risk_metrics"       : f"{RAW_DIR}/05_risk_metrics.csv",
    "fund_manager"       : f"{RAW_DIR}/06_fund_manager.csv",
    "portfolio_holdings" : f"{RAW_DIR}/07_portfolio_holdings.csv",
    "investor_data"      : f"{RAW_DIR}/08_investor_data.csv",
    "benchmark_index"    : f"{RAW_DIR}/09_benchmark_index.csv",
    "expense_ratio"      : f"{RAW_DIR}/10_expense_ratio.csv",
}

dataframes = {}
for name, path in DATASETS.items():
    if os.path.exists(path):
        df = pd.read_csv(path)
        dataframes[name] = df
        print(f"Loaded {name}: {df.shape}")
    else:
        print(f"Not found: {path}")
'''

with open('data_ingestion.py', 'w') as f:
    f.write(code)


In [29]:
code = '''import requests
import pandas as pd
import time

schemes = {
    "SBI Bluechip"     : 119551,
    "ICICI Bluechip"   : 120503,
    "Nippon Large Cap" : 118632,
    "Axis Bluechip"    : 119092,
    "Kotak Bluechip"   : 120841,
}

all_nav = []
for name, code in schemes.items():
    url      = f"https://api.mfapi.in/mf/{code}"
    response = requests.get(url)
    if response.status_code == 200:
        data = response.json()
        for record in data["data"]:
            all_nav.append({
                "scheme_code" : code,
                "scheme_name" : name,
                "date"        : record["date"],
                "nav"         : record["nav"]
            })
        print(f"Fetched {name}: {len(data['data'])} records")
    time.sleep(0.5)

df = pd.DataFrame(all_nav)
df.to_csv("data/raw/all_5_schemes_nav.csv", index=False)
print(f"Saved: {len(df)} total records")
'''

with open('live_nav_fetch.py', 'w') as f:
    f.write(code)


In [31]:
import subprocess

commands = [
    'git add .',
    'git commit -m "Day 1: Data ingestion complete"',
]

print("=" * 55)
print("   GIT COMMIT — DAY 1")
print("=" * 55)

for cmd in commands:
    result = subprocess.run(
        cmd, shell=True,
        capture_output=True, text=True
    )
    print(f"\n  Command : {cmd}")
    if result.stdout:
        print(f"  Output  : {result.stdout}")
    if result.stderr:
        print(f"  Info    : {result.stderr}")



   GIT COMMIT — DAY 1

  Command : git add .
  Info    : fatal: not a git repository (or any of the parent directories): .git


  Command : git commit -m "Day 1: Data ingestion complete"
  Info    : fatal: not a git repository (or any of the parent directories): .git



In [32]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
import sqlite3
import os
import warnings
warnings.filterwarnings('ignore')

os.makedirs('data/processed', exist_ok=True)
os.makedirs('reports', exist_ok=True)



In [33]:
df_nav = pd.read_csv('02_nav_history.csv')

print("=" * 60)
print("   CLEANING NAV HISTORY")
print("=" * 60)
print(f"  Before : {df_nav.shape}")

# 1. Parse dates
df_nav['date'] = pd.to_datetime(df_nav['date'])
print("   Dates parsed to datetime")

# 2. Sort by amfi_code + date
df_nav = df_nav.sort_values(['amfi_code','date']).reset_index(drop=True)
print("   Sorted by amfi_code + date")

# 3. Remove duplicates
before_dedup = len(df_nav)
df_nav.drop_duplicates(subset=['amfi_code','date'], inplace=True)
print(f"   Duplicates removed: {before_dedup - len(df_nav)}")

# 4. Forward fill missing NAV (holidays/weekends)
df_nav = df_nav.set_index('date')
df_nav = df_nav.groupby('amfi_code').apply(
    lambda x: x.reindex(
        pd.date_range(x.index.min(), x.index.max(), freq='D')
    ).ffill()
).reset_index(level=0, drop=True).reset_index()
df_nav.rename(columns={'index':'date'}, inplace=True)
print("   Missing NAV forward-filled for holidays/weekends")

# 5. Validate NAV > 0
invalid_nav = df_nav[df_nav['nav'] <= 0]
print(f"   Invalid NAV records (<=0): {len(invalid_nav)}")
df_nav = df_nav[df_nav['nav'] > 0]

# 6. Final check
print(f"\n  After  : {df_nav.shape}")
print(f"  Missing: {df_nav.isnull().sum().sum()}")
print(f"  Date Range: {df_nav['date'].min()} → {df_nav['date'].max()}")

# Save
df_nav.to_csv('data/processed/02_nav_history_clean.csv', index=False)
df_nav.head(3)

   CLEANING NAV HISTORY
  Before : (46000, 3)
   Dates parsed to datetime
   Sorted by amfi_code + date
   Duplicates removed: 0
   Missing NAV forward-filled for holidays/weekends
   Invalid NAV records (<=0): 0

  After  : (64320, 3)
  Missing: 0
  Date Range: 2022-01-03 00:00:00 → 2026-05-29 00:00:00


,date,amfi_code,nav
0,2022-01-03,100016.0000,520.4608
1,2022-01-04,100016.0000,515.0971
2,2022-01-05,100016.0000,521.7239


In [34]:
df_inv = pd.read_csv('08_investor_transactions.csv')

print("Investor Transactions Columns:")
for col in df_inv.columns:
    print(f"  → {col}")
print(f"\nFirst 3 rows:")
print(df_inv.head(3).to_string())
print(f"\nUnique transaction types:")
print(df_inv.iloc[:,2].value_counts() if len(df_inv.columns) > 2 else "Check columns")

Investor Transactions Columns:
  → investor_id
  → transaction_date
  → amfi_code
  → transaction_type
  → amount_inr
  → state
  → city
  → city_tier
  → age_group
  → gender
  → annual_income_lakh
  → payment_mode
  → kyc_status

First 3 rows:
  investor_id transaction_date  amfi_code transaction_type  amount_inr      state       city city_tier age_group  gender  annual_income_lakh payment_mode kyc_status
0   INV003054       2024-01-01     119092              SIP        1834  Telangana  Hyderabad       T30       56+  Female             77.1000          UPI   Verified
1   INV002952       2024-01-01     148567       Redemption      392882     Punjab   Amritsar       B30     18-25    Male              7.1000       Cheque   Verified
2   INV003420       2024-01-01     118636              SIP         912    Haryana  Faridabad       B30     36-45    Male             47.2000      Mandate   Verified

Unique transaction types:
amfi_code
148568    883
125498    877
120504    870
119095    863
1

In [35]:
df_inv = pd.read_csv('08_investor_transactions.csv')

print("=" * 60)
print("   CLEANING INVESTOR TRANSACTIONS")
print("=" * 60)
print(f"  Before : {df_inv.shape}")
print(f"  Columns: {df_inv.columns.tolist()}")


date_col = [col for col in df_inv.columns if 'date' in col.lower()][0]
df_inv[date_col] = pd.to_datetime(df_inv[date_col])
print(f"   Date column '{date_col}' parsed")


txn_col = [col for col in df_inv.columns
           if 'transaction' in col.lower() or 'type' in col.lower()][0]

print(f"\n  Before standardization:")
print(df_inv[txn_col].value_counts().to_string())

txn_map = {
    # SIP variations
    'sip'           : 'SIP',
    'SIP'           : 'SIP',
    'Sip'           : 'SIP',
    'systematic'    : 'SIP',
    'SYSTEMATIC'    : 'SIP',

    'lumpsum'       : 'Lumpsum',
    'LUMPSUM'       : 'Lumpsum',
    'Lumpsum'       : 'Lumpsum',
    'lump sum'      : 'Lumpsum',
    'one time'      : 'Lumpsum',
    'One Time'      : 'Lumpsum',
    'ONETIME'       : 'Lumpsum',

    'redemption'    : 'Redemption',
    'REDEMPTION'    : 'Redemption',
    'Redemption'    : 'Redemption',
    'redeem'        : 'Redemption',
    'REDEEM'        : 'Redemption',
    'withdrawal'    : 'Redemption',
    'WITHDRAWAL'    : 'Redemption',
}
df_inv[txn_col] = df_inv[txn_col].replace(txn_map)
df_inv[txn_col] = df_inv[txn_col].str.strip().str.title()

print(f"\n  After standardization:")
print(df_inv[txn_col].value_counts().to_string())


amt_col = [col for col in df_inv.columns
           if 'amount' in col.lower()][0]
before = len(df_inv)
df_inv = df_inv[df_inv[amt_col] > 0]
print(f"\n   Invalid amounts removed: {before - len(df_inv)}")

if 'kyc_status' in df_inv.columns:
    print(f"\n  KYC Status values:")
    print(df_inv['kyc_status'].value_counts().to_string())
    df_inv['kyc_status'] = df_inv['kyc_status'].str.strip().str.upper()
    valid_kyc = ['VERIFIED', 'PENDING', 'REJECTED', 'Y', 'N']
    invalid_kyc = df_inv[~df_inv['kyc_status'].isin(valid_kyc)]
    print(f"   Invalid KYC records: {len(invalid_kyc)}")


df_inv.drop_duplicates(inplace=True)

print(f"\n  After  : {df_inv.shape}")
print(f"  Missing: {df_inv.isnull().sum().sum()}")

df_inv.to_csv('data/processed/08_investor_transactions_clean.csv',
              index=False)
print(f"\n Saved: data/processed/08_investor_transactions_clean.csv")

   CLEANING INVESTOR TRANSACTIONS
  Before : (32778, 13)
  Columns: ['investor_id', 'transaction_date', 'amfi_code', 'transaction_type', 'amount_inr', 'state', 'city', 'city_tier', 'age_group', 'gender', 'annual_income_lakh', 'payment_mode', 'kyc_status']
   Date column 'transaction_date' parsed

  Before standardization:
transaction_date
2025-04-28    88
2025-04-25    87
2024-03-13    86
2024-06-12    86
2024-10-16    85
2024-01-10    84
2025-05-19    83
2024-04-15    82
2024-05-24    81
2024-01-27    80
2025-01-07    80
2024-12-14    80
2024-06-23    80
2024-02-03    80
2025-03-12    79
2024-06-18    79
2025-01-05    79
2024-04-26    79
2024-01-31    78
2024-11-01    78
2024-07-30    78
2025-03-05    78
2024-08-15    78
2024-10-12    78
2024-07-13    78
2024-08-26    78
2024-06-17    77
2024-02-13    77
2025-01-01    77
2024-11-09    77
2025-05-15    77
2024-04-05    77
2024-11-07    77
2024-06-09    76
2025-01-16    76
2024-04-12    76
2024-07-10    76
2025-02-27    76
2025-05-23   

AttributeError: Can only use .str accessor with string values!

In [36]:
df_sp = pd.read_csv('07_scheme_performance.csv')

print("Scheme Performance Columns:")
for col in df_sp.columns:
    print(f"  → {col}")
print(f"\nFirst 3 rows:")
print(df_sp.head(3).to_string())

Scheme Performance Columns:
  → amfi_code
  → scheme_name
  → fund_house
  → category
  → plan
  → return_1yr_pct
  → return_3yr_pct
  → return_5yr_pct
  → benchmark_3yr_pct
  → alpha
  → beta
  → sharpe_ratio
  → sortino_ratio
  → std_dev_ann_pct
  → max_drawdown_pct
  → aum_crore
  → expense_ratio_pct
  → morningstar_rating
  → risk_grade

First 3 rows:
   amfi_code                                 scheme_name       fund_house   category     plan  return_1yr_pct  return_3yr_pct  return_5yr_pct  benchmark_3yr_pct  alpha   beta  sharpe_ratio  sortino_ratio  std_dev_ann_pct  max_drawdown_pct  aum_crore  expense_ratio_pct  morningstar_rating risk_grade
0     119551   SBI Bluechip Fund - Regular Plan - Growth  SBI Mutual Fund  Large Cap  Regular         12.4200         12.3600         14.4500            11.4900 0.8700 0.8900        0.8800         1.2900          14.0000          -21.7000      14288             1.5400                   4   Moderate
1     119552    SBI Bluechip Fund - Direct

In [38]:
df_sp = pd.read_csv('07_scheme_performance.csv')

print("=" * 60)
print("   CLEANING SCHEME PERFORMANCE")
print("=" * 60)
print(f"  Before : {df_sp.shape}")

# 1. Get return columns
return_cols = [col for col in df_sp.columns
               if 'return' in col.lower()
               or 'cagr' in col.lower()
               or 'pct' in col.lower()
               or 'perf' in col.lower()]
print(f"\n  Return columns found: {return_cols}")

# 2. Validate all return values are numeric
for col in return_cols:
    df_sp[col] = pd.to_numeric(df_sp[col], errors='coerce')
    non_numeric = df_sp[col].isnull().sum()
    print(f"   {col}: {non_numeric} non-numeric values coerced")

# 3. Flag anomalies (returns > 200% or < -100%)
anomaly_flags = []
for col in return_cols:
    anomalies = df_sp[
        (df_sp[col] > 200) | (df_sp[col] < -100)
    ]
    if len(anomalies) > 0:
        anomaly_flags.append(f"{col}: {len(anomalies)} anomalies")
        print(f"    {col}: {len(anomalies)} anomalies flagged")

df_sp['has_anomaly'] = False
for col in return_cols:
    df_sp.loc[
        (df_sp[col] > 200) | (df_sp[col] < -100),
        'has_anomaly'
    ] = True

print(f"\n  Total anomaly rows flagged: {df_sp['has_anomaly'].sum()}")

# 4. Check expense_ratio range (0.1% - 2.5%)
exp_col = [col for col in df_sp.columns
           if 'expense' in col.lower()]
if exp_col:
    exp_col = exp_col[0]
    df_sp[exp_col] = pd.to_numeric(
        df_sp[exp_col], errors='coerce')
    out_of_range = df_sp[
        (df_sp[exp_col] < 0.1) |
        (df_sp[exp_col] > 2.5)
    ]
    print(f"\n  Expense ratio out of range (0.1-2.5%): {len(out_of_range)}")
    print(f"  Expense ratio stats:")
    print(df_sp[exp_col].describe().round(4).to_string())

# 5. Remove duplicates
df_sp.drop_duplicates(inplace=True)

print(f"\n  After  : {df_sp.shape}")
print(f"  Missing: {df_sp.isnull().sum().sum()}")

df_sp.to_csv('data/processed/07_scheme_performance_clean.csv',
             index=False)


   CLEANING SCHEME PERFORMANCE
  Before : (40, 19)

  Return columns found: ['return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct', 'benchmark_3yr_pct', 'std_dev_ann_pct', 'max_drawdown_pct', 'expense_ratio_pct']
   return_1yr_pct: 0 non-numeric values coerced
   return_3yr_pct: 0 non-numeric values coerced
   return_5yr_pct: 0 non-numeric values coerced
   benchmark_3yr_pct: 0 non-numeric values coerced
   std_dev_ann_pct: 0 non-numeric values coerced
   max_drawdown_pct: 0 non-numeric values coerced
   expense_ratio_pct: 0 non-numeric values coerced

  Total anomaly rows flagged: 0

  Expense ratio out of range (0.1-2.5%): 0
  Expense ratio stats:
count   40.0000
mean     1.2370
std      0.3866
min      0.5500
25%      0.7875
50%      1.4250
75%      1.5400
max      1.6400

  After  : (40, 20)
  Missing: 0


In [39]:
import pandas as pd
import os

remaining = {
    '01_fund_master.csv'        : '01_fund_master_clean.csv',
    '03_aum_by_fund_house.csv'  : '03_aum_by_fund_house_clean.csv',
    '04_monthly_sip_inflows.csv': '04_monthly_sip_inflows_clean.csv',
    '05_category_inflows.csv'   : '05_category_inflows_clean.csv',
    '06_industry_folio_count.csv': '06_industry_folio_count_clean.csv',
    '09_portfolio_holdings.csv' : '09_portfolio_holdings_clean.csv',
    '10_benchmark_indices.csv'  : '10_benchmark_indices_clean.csv',
}

print("=" * 60)
print("   CLEANING REMAINING CSVs")
print("=" * 60)

for src, dst in remaining.items():
    if os.path.exists(src):
        df = pd.read_csv(src)
        before = df.shape

        # Parse date columns
        for col in df.columns:
            if 'date' in col.lower() or 'month' in col.lower():
                try:
                    df[col] = pd.to_datetime(df[col])
                except:
                    pass

        # Remove duplicates
        df.drop_duplicates(inplace=True)

        # Convert numeric columns
        for col in df.columns:
            if df[col].dtype == 'object':
                try:
                    df[col] = pd.to_numeric(df[col])
                except:
                    pass

        after = df.shape
        df.to_csv(f'data/processed/{dst}', index=False)
        print(f"   {src}")
        print(f"     Before: {before} → After: {after}")
    else:
        print(f"   {src} not found!")



   CLEANING REMAINING CSVs
   01_fund_master.csv
     Before: (40, 15) → After: (40, 15)
   03_aum_by_fund_house.csv
     Before: (90, 5) → After: (90, 5)
   04_monthly_sip_inflows.csv
     Before: (48, 6) → After: (48, 6)
   05_category_inflows.csv
     Before: (144, 3) → After: (144, 3)
   06_industry_folio_count.csv
     Before: (21, 6) → After: (21, 6)
   09_portfolio_holdings.csv
     Before: (322, 8) → After: (322, 8)
   10_benchmark_indices.csv
     Before: (8050, 3) → After: (8050, 3)


In [40]:
schema_sql = """

-- BLUESTOCK MUTUAL FUND STAR SCHEMA

-- Dimension: Fund
CREATE TABLE IF NOT EXISTS dim_fund (
    amfi_code          INTEGER PRIMARY KEY,
    fund_house         TEXT    NOT NULL,
    scheme_name        TEXT    NOT NULL,
    category           TEXT,
    sub_category       TEXT,
    plan               TEXT,
    benchmark          TEXT,
    fund_manager       TEXT,
    risk_category      TEXT,
    sebi_category_code TEXT,
    expense_ratio_pct  REAL,
    exit_load_pct      REAL,
    min_sip_amount     REAL,
    min_lumpsum_amount REAL,
    launch_date        DATE
);

-- Dimension: Date
CREATE TABLE IF NOT EXISTS dim_date (
    date_id     INTEGER PRIMARY KEY,
    full_date   DATE    NOT NULL UNIQUE,
    day         INTEGER,
    month       INTEGER,
    month_name  TEXT,
    quarter     INTEGER,
    year        INTEGER,
    is_weekend  INTEGER,
    is_monthend INTEGER
);

-- Fact: NAV History
CREATE TABLE IF NOT EXISTS fact_nav (
    nav_id    INTEGER PRIMARY KEY AUTOINCREMENT,
    amfi_code INTEGER NOT NULL,
    date      DATE    NOT NULL,
    nav       REAL    NOT NULL,
    FOREIGN KEY (amfi_code) REFERENCES dim_fund(amfi_code)
);

-- Fact: Transactions
CREATE TABLE IF NOT EXISTS fact_transactions (
    txn_id           INTEGER PRIMARY KEY AUTOINCREMENT,
    amfi_code        INTEGER,
    transaction_date DATE,
    transaction_type TEXT,
    amount           REAL,
    units            REAL,
    nav_at_txn       REAL,
    investor_id      TEXT,
    state            TEXT,
    kyc_status       TEXT,
    FOREIGN KEY (amfi_code) REFERENCES dim_fund(amfi_code)
);

-- Fact: Performance
CREATE TABLE IF NOT EXISTS fact_performance (
    perf_id          INTEGER PRIMARY KEY AUTOINCREMENT,
    amfi_code        INTEGER,
    as_of_date       DATE,
    return_1m        REAL,
    return_3m        REAL,
    return_6m        REAL,
    return_1y        REAL,
    return_3y        REAL,
    return_5y        REAL,
    expense_ratio    REAL,
    has_anomaly      INTEGER DEFAULT 0,
    FOREIGN KEY (amfi_code) REFERENCES dim_fund(amfi_code)
);

-- Fact: AUM
CREATE TABLE IF NOT EXISTS fact_aum (
    aum_id      INTEGER PRIMARY KEY AUTOINCREMENT,
    amfi_code   INTEGER,
    fund_house  TEXT,
    month       DATE,
    aum_crores  REAL,
    FOREIGN KEY (amfi_code) REFERENCES dim_fund(amfi_code)
);
"""

with open('sql/schema.sql', 'w') as f:
    f.write(schema_sql)

print(schema_sql)



-- BLUESTOCK MUTUAL FUND STAR SCHEMA

-- Dimension: Fund
CREATE TABLE IF NOT EXISTS dim_fund (
    amfi_code          INTEGER PRIMARY KEY,
    fund_house         TEXT    NOT NULL,
    scheme_name        TEXT    NOT NULL,
    category           TEXT,
    sub_category       TEXT,
    plan               TEXT,
    benchmark          TEXT,
    fund_manager       TEXT,
    risk_category      TEXT,
    sebi_category_code TEXT,
    expense_ratio_pct  REAL,
    exit_load_pct      REAL,
    min_sip_amount     REAL,
    min_lumpsum_amount REAL,
    launch_date        DATE
);

-- Dimension: Date
CREATE TABLE IF NOT EXISTS dim_date (
    date_id     INTEGER PRIMARY KEY,
    full_date   DATE    NOT NULL UNIQUE,
    day         INTEGER,
    month       INTEGER,
    month_name  TEXT,
    quarter     INTEGER,
    year        INTEGER,
    is_weekend  INTEGER,
    is_monthend INTEGER
);

-- Fact: NAV History
CREATE TABLE IF NOT EXISTS fact_nav (
    nav_id    INTEGER PRIMARY KEY AUTOINCREMENT,
    amfi

In [42]:
import pandas as pd
from sqlalchemy import create_engine, text
import os

engine = create_engine('sqlite:///bluestock_mf.db')

print("=" * 60)
print("   LOADING DATA INTO SQLITE")
print("=" * 60)


def load_table(csv_path, table_name, engine):
    # Try processed first, then original
    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        df.to_sql(table_name, engine,
                  if_exists='replace', index=False)
        print(f"   {table_name:<25} : {len(df):,} rows")
        return df
    else:
        print(f"   {table_name:<25} : File not found — {csv_path}")
        return None




path = 'data/processed/01_fund_master_clean.csv'
if not os.path.exists(path):
    path = '01_fund_master.csv'
load_table(path, 'dim_fund', engine)


path = 'data/processed/02_nav_history_clean.csv'
if not os.path.exists(path):
    path = '02_nav_history.csv'
load_table(path, 'fact_nav', engine)


path = 'data/processed/08_investor_transactions_clean.csv'
if not os.path.exists(path):
    path = '08_investor_transactions.csv'
load_table(path, 'fact_transactions', engine)


path = 'data/processed/07_scheme_performance_clean.csv'
if not os.path.exists(path):
    path = '07_scheme_performance.csv'
load_table(path, 'fact_performance', engine)


path = 'data/processed/03_aum_by_fund_house_clean.csv'
if not os.path.exists(path):
    path = '03_aum_by_fund_house.csv'
load_table(path, 'fact_aum', engine)


extra = {
    '04_monthly_sip_inflows.csv'  : 'sip_inflows',
    '05_category_inflows.csv'     : 'category_inflows',
    '06_industry_folio_count.csv' : 'folio_count',
    '09_portfolio_holdings.csv'   : 'portfolio_holdings',
    '10_benchmark_indices.csv'    : 'benchmark_indices',
}
for csv, table in extra.items():
    proc = f'data/processed/{csv.replace(".csv","_clean.csv")}'
    path = proc if os.path.exists(proc) else csv
    load_table(path, table, engine)


print(f"\n{'='*60}")
print("   VERIFICATION — ROW COUNTS IN DB")
print(f"{'='*60}")

with engine.connect() as conn:
    result = conn.execute(
        text("SELECT name FROM sqlite_master WHERE type='table'"))
    tables = [row[0] for row in result]
    for table in tables:
        count = conn.execute(
            text(f"SELECT COUNT(*) FROM {table}")
        ).fetchone()[0]
        print(f"   {table:<30} : {count:,} rows")

print(f"\n SQLite DB ready: bluestock_mf.db")
print(f"   Size: {os.path.getsize('bluestock_mf.db')/1024:.1f} KB")

   LOADING DATA INTO SQLITE
   dim_fund                  : 40 rows
   fact_nav                  : 64,320 rows
   fact_transactions         : 32,778 rows
   fact_performance          : 40 rows
   fact_aum                  : 90 rows
   sip_inflows               : 48 rows
   category_inflows          : 144 rows
   folio_count               : 21 rows
   portfolio_holdings        : 322 rows
   benchmark_indices         : 8,050 rows

   VERIFICATION — ROW COUNTS IN DB
   dim_fund                       : 40 rows
   fact_nav                       : 64,320 rows
   fact_transactions              : 32,778 rows
   fact_performance               : 40 rows
   fact_aum                       : 90 rows
   sip_inflows                    : 48 rows
   category_inflows               : 144 rows
   folio_count                    : 21 rows
   portfolio_holdings             : 322 rows
   benchmark_indices              : 8,050 rows

 SQLite DB ready: bluestock_mf.db
   Size: 5624.0 KB


In [1]:
import pandas as pd
from sqlalchemy import create_engine, text

engine = create_engine('sqlite:///bluestock_mf.db')

queries = {
    "Q1: Top 5 Fund Houses by AUM": """
        SELECT
            fund_house,
            ROUND(SUM(aum_crores), 2) AS total_aum_crores
        FROM fact_aum
        GROUP BY fund_house
        ORDER BY total_aum_crores DESC
        LIMIT 5
    """,

    "Q2: Average NAV Per Month": """
        SELECT
            STRFTIME('%Y-%m', date) AS month,
            amfi_code,
            ROUND(AVG(nav), 4)      AS avg_nav
        FROM fact_nav
        GROUP BY month, amfi_code
        ORDER BY month DESC
        LIMIT 10
    """,

    "Q3: SIP Year-on-Year Growth": """
        SELECT
            STRFTIME('%Y', transaction_date) AS year,
            COUNT(*)                          AS sip_count,
            ROUND(SUM(amount), 2)             AS total_sip_amount
        FROM fact_transactions
        WHERE UPPER(transaction_type) = 'SIP'
        GROUP BY year
        ORDER BY year
    """,

    "Q4: Transactions by State": """
        SELECT
            state,
            COUNT(*)              AS total_transactions,
            ROUND(SUM(amount), 2) AS total_amount
        FROM fact_transactions
        GROUP BY state
        ORDER BY total_transactions DESC
        LIMIT 10
    """,

    "Q5: Funds with Expense Ratio < 1%": """
        SELECT
            scheme_name,
            fund_house,
            category,
            expense_ratio_pct
        FROM dim_fund
        WHERE expense_ratio_pct < 1.0
        ORDER BY expense_ratio_pct ASC
        LIMIT 10
    """,

    "Q6: Top 10 NAV Growth Funds": """
        SELECT
            f.scheme_name,
            f.fund_house,
            f.category,
            ROUND(MIN(n.nav), 2)                   AS start_nav,
            ROUND(MAX(n.nav), 2)                   AS end_nav,
            ROUND((MAX(n.nav)-MIN(n.nav))
                  /MIN(n.nav)*100, 2)              AS growth_pct
        FROM fact_nav n
        JOIN dim_fund f ON n.amfi_code = f.amfi_code
        GROUP BY n.amfi_code
        ORDER BY growth_pct DESC
        LIMIT 10
    """,

    "Q7: Monthly SIP Trend": """
        SELECT
            STRFTIME('%Y-%m', transaction_date) AS month,
            COUNT(*)                             AS sip_count,
            ROUND(SUM(amount), 2)               AS total_sip
        FROM fact_transactions
        WHERE UPPER(transaction_type) = 'SIP'
        GROUP BY month
        ORDER BY month
        LIMIT 12
    """,

    "Q8: Category-wise Transaction Volume": """
        SELECT
            f.category,
            COUNT(t.rowid)        AS txn_count,
            ROUND(SUM(t.amount),2) AS total_amount
        FROM fact_transactions t
        JOIN dim_fund f ON t.amfi_code = f.amfi_code
        GROUP BY f.category
        ORDER BY total_amount DESC
    """,

    "Q9: Risk Category vs Avg Expense Ratio": """
        SELECT
            risk_category,
            COUNT(*)                           AS scheme_count,
            ROUND(AVG(expense_ratio_pct), 4)   AS avg_expense_ratio,
            ROUND(MIN(expense_ratio_pct), 4)   AS min_expense,
            ROUND(MAX(expense_ratio_pct), 4)   AS max_expense
        FROM dim_fund
        GROUP BY risk_category
        ORDER BY avg_expense_ratio
    """,

    "Q10: Top Fund Managers by Scheme Count": """
        SELECT
            fund_manager,
            COUNT(*)                          AS scheme_count,
            ROUND(AVG(expense_ratio_pct), 4)  AS avg_expense,
            GROUP_CONCAT(DISTINCT category)   AS categories
        FROM dim_fund
        GROUP BY fund_manager
        ORDER BY scheme_count DESC
        LIMIT 10
    """
}

print("=" * 60)
print("   10 ANALYTICAL SQL QUERIES — RESULTS")
print("=" * 60)

for title, query in queries.items():
    print(f"\n{'─'*60}")
    print(f"  {title}")
    print(f"{'─'*60}")
    try:
        with engine.connect() as conn:
            df_result = pd.read_sql(query, conn)
        print(df_result.to_string(index=False))
        print(f"  → {len(df_result)} rows returned")
    except Exception as e:
        print(f"    Error: {e}")

print(f"\n{'='*60}")


   10 ANALYTICAL SQL QUERIES — RESULTS

────────────────────────────────────────────────────────────
  Q1: Top 5 Fund Houses by AUM
────────────────────────────────────────────────────────────
    Error: (sqlite3.OperationalError) no such column: aum_crores
[SQL: 
        SELECT
            fund_house,
            ROUND(SUM(aum_crores), 2) AS total_aum_crores
        FROM fact_aum
        GROUP BY fund_house
        ORDER BY total_aum_crores DESC
        LIMIT 5
    ]
(Background on this error at: https://sqlalche.me/e/20/e3q8)

────────────────────────────────────────────────────────────
  Q2: Average NAV Per Month
────────────────────────────────────────────────────────────
  month  amfi_code  avg_nav
2026-05   100016.0 592.3666
2026-05   100025.0  31.9019
2026-05   100033.0 352.1477
2026-05   101206.0 780.4739
2026-05   101207.0  53.1598
2026-05   101208.0 408.8293
2026-05   102885.0 184.5136
2026-05   102886.0 125.4806
2026-05   102887.0 366.2445
2026-05   118632.0 109.5629
  → 10 

In [2]:
queries_sql = """-- 
-- BLUESTOCK MF - 10 ANALYTICAL SQL QUERIES

-- Q1: Top 5 Fund Houses by AUM
SELECT
    fund_house,
    ROUND(SUM(aum_crores), 2) AS total_aum_crores
FROM fact_aum
GROUP BY fund_house
ORDER BY total_aum_crores DESC
LIMIT 5;

-- Q2: Average NAV Per Month
SELECT
    STRFTIME('%Y-%m', date) AS month,
    amfi_code,
    ROUND(AVG(nav), 4)      AS avg_nav
FROM fact_nav
GROUP BY month, amfi_code
ORDER BY month DESC
LIMIT 10;

-- Q3: SIP Year-on-Year Growth
SELECT
    STRFTIME('%Y', transaction_date) AS year,
    COUNT(*)                          AS sip_count,
    ROUND(SUM(amount), 2)             AS total_sip_amount
FROM fact_transactions
WHERE UPPER(transaction_type) = 'SIP'
GROUP BY year
ORDER BY year;

-- Q4: Transactions by State
SELECT
    state,
    COUNT(*)              AS total_transactions,
    ROUND(SUM(amount), 2) AS total_amount
FROM fact_transactions
GROUP BY state
ORDER BY total_transactions DESC
LIMIT 10;

-- Q5: Funds with Expense Ratio < 1%
SELECT
    scheme_name,
    fund_house,
    category,
    expense_ratio_pct
FROM dim_fund
WHERE expense_ratio_pct < 1.0
ORDER BY expense_ratio_pct ASC
LIMIT 10;

-- Q6: Top 10 NAV Growth Funds
SELECT
    f.scheme_name,
    f.fund_house,
    f.category,
    ROUND(MIN(n.nav), 2)                AS start_nav,
    ROUND(MAX(n.nav), 2)                AS end_nav,
    ROUND((MAX(n.nav)-MIN(n.nav))
          /MIN(n.nav)*100, 2)           AS growth_pct
FROM fact_nav n
JOIN dim_fund f ON n.amfi_code = f.amfi_code
GROUP BY n.amfi_code
ORDER BY growth_pct DESC
LIMIT 10;

-- Q7: Monthly SIP Trend
SELECT
    STRFTIME('%Y-%m', transaction_date) AS month,
    COUNT(*)                             AS sip_count,
    ROUND(SUM(amount), 2)               AS total_sip
FROM fact_transactions
WHERE UPPER(transaction_type) = 'SIP'
GROUP BY month
ORDER BY month
LIMIT 12;

-- Q8: Category-wise Transaction Volume
SELECT
    f.category,
    COUNT(t.rowid)         AS txn_count,
    ROUND(SUM(t.amount),2) AS total_amount
FROM fact_transactions t
JOIN dim_fund f ON t.amfi_code = f.amfi_code
GROUP BY f.category
ORDER BY total_amount DESC;

-- Q9: Risk Category vs Avg Expense Ratio
SELECT
    risk_category,
    COUNT(*)                          AS scheme_count,
    ROUND(AVG(expense_ratio_pct), 4)  AS avg_expense_ratio,
    ROUND(MIN(expense_ratio_pct), 4)  AS min_expense,
    ROUND(MAX(expense_ratio_pct), 4)  AS max_expense
FROM dim_fund
GROUP BY risk_category
ORDER BY avg_expense_ratio;

-- Q10: Top Fund Managers by Scheme Count
SELECT
    fund_manager,
    COUNT(*)                          AS scheme_count,
    ROUND(AVG(expense_ratio_pct), 4)  AS avg_expense,
    GROUP_CONCAT(DISTINCT category)   AS categories
FROM dim_fund
GROUP BY fund_manager
ORDER BY scheme_count DESC
LIMIT 10;
"""

import os
os.makedirs('sql', exist_ok=True)
with open('sql/queries.sql', 'w') as f:
    f.write(queries_sql)



In [3]:
schema_sql = """-- 
-- BLUESTOCK MUTUAL FUND STAR SCHEMA
-- 

-- Dimension: Fund
CREATE TABLE IF NOT EXISTS dim_fund (
    amfi_code          INTEGER PRIMARY KEY,
    fund_house         TEXT    NOT NULL,
    scheme_name        TEXT    NOT NULL,
    category           TEXT,
    sub_category       TEXT,
    plan               TEXT,
    benchmark          TEXT,
    fund_manager       TEXT,
    risk_category      TEXT,
    sebi_category_code TEXT,
    expense_ratio_pct  REAL,
    exit_load_pct      REAL,
    min_sip_amount     REAL,
    min_lumpsum_amount REAL,
    launch_date        DATE
);

-- Dimension: Date
CREATE TABLE IF NOT EXISTS dim_date (
    date_id     INTEGER PRIMARY KEY,
    full_date   DATE    NOT NULL UNIQUE,
    day         INTEGER,
    month       INTEGER,
    month_name  TEXT,
    quarter     INTEGER,
    year        INTEGER,
    is_weekend  INTEGER,
    is_monthend INTEGER
);

-- Fact: NAV History
CREATE TABLE IF NOT EXISTS fact_nav (
    nav_id    INTEGER PRIMARY KEY AUTOINCREMENT,
    amfi_code INTEGER NOT NULL,
    date      DATE    NOT NULL,
    nav       REAL    NOT NULL,
    FOREIGN KEY (amfi_code) REFERENCES dim_fund(amfi_code)
);

-- Fact: Transactions
CREATE TABLE IF NOT EXISTS fact_transactions (
    txn_id           INTEGER PRIMARY KEY AUTOINCREMENT,
    amfi_code        INTEGER,
    transaction_date DATE,
    transaction_type TEXT,
    amount           REAL,
    units            REAL,
    nav_at_txn       REAL,
    investor_id      TEXT,
    state            TEXT,
    kyc_status       TEXT,
    FOREIGN KEY (amfi_code) REFERENCES dim_fund(amfi_code)
);

-- Fact: Performance
CREATE TABLE IF NOT EXISTS fact_performance (
    perf_id       INTEGER PRIMARY KEY AUTOINCREMENT,
    amfi_code     INTEGER,
    as_of_date    DATE,
    return_1m     REAL,
    return_3m     REAL,
    return_6m     REAL,
    return_1y     REAL,
    return_3y     REAL,
    return_5y     REAL,
    expense_ratio REAL,
    has_anomaly   INTEGER DEFAULT 0,
    FOREIGN KEY (amfi_code) REFERENCES dim_fund(amfi_code)
);

-- Fact: AUM
CREATE TABLE IF NOT EXISTS fact_aum (
    aum_id     INTEGER PRIMARY KEY AUTOINCREMENT,
    amfi_code  INTEGER,
    fund_house TEXT,
    month      DATE,
    aum_crores REAL,
    FOREIGN KEY (amfi_code) REFERENCES dim_fund(amfi_code)
);
"""

with open('sql/schema.sql', 'w') as f:
    f.write(schema_sql)



In [4]:
import os

files = {
    'data/processed/01_fund_master_clean.csv'          : 'Fund Master Cleaned',
    'data/processed/02_nav_history_clean.csv'          : 'NAV History Cleaned',
    'data/processed/03_aum_by_fund_house_clean.csv'    : 'AUM Cleaned',
    'data/processed/04_monthly_sip_inflows_clean.csv'  : 'SIP Inflows Cleaned',
    'data/processed/05_category_inflows_clean.csv'     : 'Category Inflows Cleaned',
    'data/processed/06_industry_folio_count_clean.csv' : 'Folio Count Cleaned',
    'data/processed/07_scheme_performance_clean.csv'   : 'Scheme Performance Cleaned',
    'data/processed/08_investor_transactions_clean.csv': 'Investor Transactions Cleaned',
    'data/processed/09_portfolio_holdings_clean.csv'   : 'Portfolio Holdings Cleaned',
    'data/processed/10_benchmark_indices_clean.csv'    : 'Benchmark Indices Cleaned',
    'bluestock_mf.db'                                  : 'SQLite Database',
    'sql/schema.sql'                                   : 'Schema SQL',
    'sql/queries.sql'                                  : 'Queries SQL',
    'reports/data_dictionary.md'                       : 'Data Dictionary',
    'reports/data_quality_summary.txt'                 : 'Data Quality Summary',
}

print("=" * 60)
print("   DAY 2 DELIVERABLES CHECK")
print("=" * 60)

ready   = []
missing = []

for path, name in files.items():
    if os.path.exists(path):
        size = os.path.getsize(path) / 1024
        print(f"   {name:<40} ({size:.1f} KB)")
        ready.append(path)
    else:
        print(f"   {name:<40} MISSING!")
        missing.append(path)

print(f"\n  Ready   : {len(ready)}")
print(f"  Missing : {len(missing)}")

if missing:
    print(f"\n  Missing files:")
    for m in missing:
        print(f"     {m}")

   DAY 2 DELIVERABLES CHECK
   Fund Master Cleaned                      (6.7 KB)
   NAV History Cleaned                      (1853.7 KB)
   AUM Cleaned                              (4.0 KB)
   SIP Inflows Cleaned                      (1.8 KB)
   Category Inflows Cleaned                 (4.1 KB)
   Folio Count Cleaned                      (0.9 KB)
   Scheme Performance Cleaned               (6.7 KB)
   Investor Transactions Cleaned            MISSING!
   Portfolio Holdings Cleaned               (23.7 KB)
   Benchmark Indices Cleaned                (253.0 KB)
   SQLite Database                          (5624.0 KB)
   Schema SQL                               (2.3 KB)
   Queries SQL                              (2.8 KB)
   Data Dictionary                          MISSING!
   Data Quality Summary                     (0.8 KB)

  Ready   : 13
  Missing : 2

  Missing files:
     data/processed/08_investor_transactions_clean.csv
     reports/data_dictionary.md


In [5]:
import pandas as pd
import os

df_inv = pd.read_csv('08_investor_transactions.csv')

print("Columns:", df_inv.columns.tolist())
print(f"Shape  : {df_inv.shape}")
print(df_inv.head(3).to_string())

Columns: ['investor_id', 'transaction_date', 'amfi_code', 'transaction_type', 'amount_inr', 'state', 'city', 'city_tier', 'age_group', 'gender', 'annual_income_lakh', 'payment_mode', 'kyc_status']
Shape  : (32778, 13)
  investor_id transaction_date  amfi_code transaction_type  amount_inr      state       city city_tier age_group  gender  annual_income_lakh payment_mode kyc_status
0   INV003054       2024-01-01     119092              SIP        1834  Telangana  Hyderabad       T30       56+  Female                77.1          UPI   Verified
1   INV002952       2024-01-01     148567       Redemption      392882     Punjab   Amritsar       B30     18-25    Male                 7.1       Cheque   Verified
2   INV003420       2024-01-01     118636              SIP         912    Haryana  Faridabad       B30     36-45    Male                47.2      Mandate   Verified


In [6]:
import pandas as pd
import os

df_inv = pd.read_csv('08_investor_transactions.csv')

print("=" * 55)
print("   CLEANING INVESTOR TRANSACTIONS")
print("=" * 55)
print(f"  Before : {df_inv.shape}")

# 1. Parse date columns
for col in df_inv.columns:
    if 'date' in col.lower():
        df_inv[col] = pd.to_datetime(df_inv[col],
                                      errors='coerce')
        print(f"   Date parsed: {col}")

# 2. Standardize transaction_type
txn_cols = [col for col in df_inv.columns
            if 'type' in col.lower()
            or 'transaction' in col.lower()]

if txn_cols:
    txn_col = txn_cols[0]
    print(f"\n  Transaction column: {txn_col}")
    print(f"  Before:\n{df_inv[txn_col].value_counts().to_string()}")

    txn_map = {
        'sip'        : 'SIP',
        'SIP'        : 'SIP',
        'Sip'        : 'SIP',
        'systematic' : 'SIP',
        'lumpsum'    : 'Lumpsum',
        'LUMPSUM'    : 'Lumpsum',
        'Lumpsum'    : 'Lumpsum',
        'one time'   : 'Lumpsum',
        'One Time'   : 'Lumpsum',
        'onetime'    : 'Lumpsum',
        'redemption' : 'Redemption',
        'REDEMPTION' : 'Redemption',
        'Redemption' : 'Redemption',
        'redeem'     : 'Redemption',
        'withdrawal' : 'Redemption',
    }
    df_inv[txn_col] = df_inv[txn_col].replace(txn_map)
    df_inv[txn_col] = df_inv[txn_col].str.strip().str.title()
    print(f"\n  After:\n{df_inv[txn_col].value_counts().to_string()}")

# 3. Validate amount > 0
amt_cols = [col for col in df_inv.columns
            if 'amount' in col.lower()
            or 'amt' in col.lower()]

if amt_cols:
    amt_col = amt_cols[0]
    before  = len(df_inv)
    df_inv  = df_inv[df_inv[amt_col] > 0]
    print(f"\n  Invalid amounts removed: {before - len(df_inv)}")

# 4. Check KYC status
kyc_cols = [col for col in df_inv.columns
            if 'kyc' in col.lower()]
if kyc_cols:
    kyc_col = kyc_cols[0]
    df_inv[kyc_col] = df_inv[kyc_col].str.strip().str.upper()
    print(f"\n  KYC Status:")
    print(df_inv[kyc_col].value_counts().to_string())

# 5. Remove duplicates
before = len(df_inv)
df_inv.drop_duplicates(inplace=True)
print(f"\n   Duplicates removed: {before - len(df_inv)}")

print(f"\n  After  : {df_inv.shape}")
print(f"  Missing: {df_inv.isnull().sum().sum()}")

# Save
os.makedirs('data/processed', exist_ok=True)
df_inv.to_csv(
    'data/processed/08_investor_transactions_clean.csv',
    index=False)


   CLEANING INVESTOR TRANSACTIONS
  Before : (32778, 13)
   Date parsed: transaction_date

  Transaction column: transaction_date
  Before:
transaction_date
2025-04-28    88
2025-04-25    87
2024-03-13    86
2024-06-12    86
2024-10-16    85
2024-01-10    84
2025-05-19    83
2024-04-15    82
2024-05-24    81
2024-01-27    80
2025-01-07    80
2024-12-14    80
2024-06-23    80
2024-02-03    80
2025-03-12    79
2024-06-18    79
2025-01-05    79
2024-04-26    79
2024-01-31    78
2024-11-01    78
2024-07-30    78
2025-03-05    78
2024-08-15    78
2024-10-12    78
2024-07-13    78
2024-08-26    78
2024-06-17    77
2024-02-13    77
2025-01-01    77
2024-11-09    77
2025-05-15    77
2024-04-05    77
2024-11-07    77
2024-06-09    76
2025-01-16    76
2024-04-12    76
2024-07-10    76
2025-02-27    76
2025-05-23    76
2024-04-14    76
2025-03-24    76
2024-11-12    75
2025-03-29    75
2024-02-12    75
2024-04-19    75
2024-06-04    75
2024-06-24    75
2025-01-02    75
2025-03-14    75
2024-12-20

AttributeError: Can only use .str accessor with string values!

In [7]:
import os

os.makedirs('reports', exist_ok=True)

data_dict = (
"# Data Dictionary — Bluestock Mutual Fund Analytics\n"
"\n"
"## Project Overview\n"
"**Project**: Mutual Fund Customer Behavior Analytics\n"
"**Database**: bluestock_mf.db (SQLite)\n"
"**Analyst**: MERLIN J\n"
"**Date**: June 2026\n"
"\n"
"---\n"
"\n"
"## 1. dim_fund (Fund Master)\n"
"**Source**: 01_fund_master.csv\n"
"\n"
"| Column | Data Type | Description | Example |\n"
"|--------|-----------|-------------|---------|\n"
"| amfi_code | INTEGER | Unique AMFI code (PK) | 119551 |\n"
"| fund_house | TEXT | AMC name | SBI Mutual Fund |\n"
"| scheme_name | TEXT | Full scheme name | SBI Bluechip Fund |\n"
"| category | TEXT | SEBI category | Equity |\n"
"| sub_category | TEXT | Sub-category | Large Cap |\n"
"| plan | TEXT | Regular or Direct | Direct |\n"
"| launch_date | DATE | Scheme launch date | 2006-02-14 |\n"
"| benchmark | TEXT | Benchmark index | NIFTY 100 TRI |\n"
"| expense_ratio_pct | REAL | Annual fee % | 0.66 |\n"
"| exit_load_pct | REAL | Exit load % | 1.0 |\n"
"| min_sip_amount | REAL | Min SIP amount | 500 |\n"
"| min_lumpsum_amount | REAL | Min lumpsum | 1000 |\n"
"| fund_manager | TEXT | Fund manager name | Sohini Andani |\n"
"| risk_category | TEXT | Risk level | Moderate |\n"
"| sebi_category_code | TEXT | SEBI code | EC01 |\n"
"\n"
"---\n"
"\n"
"## 2. fact_nav (NAV History)\n"
"**Source**: 02_nav_history.csv\n"
"\n"
"| Column | Data Type | Description | Example |\n"
"|--------|-----------|-------------|---------|\n"
"| amfi_code | INTEGER | AMFI code (FK) | 119551 |\n"
"| date | DATE | NAV date | 2022-01-03 |\n"
"| nav | REAL | Net Asset Value | 54.3856 |\n"
"\n"
"---\n"
"\n"
"## 3. fact_transactions\n"
"**Source**: 08_investor_transactions.csv\n"
"\n"
"| Column | Data Type | Description | Example |\n"
"|--------|-----------|-------------|---------|\n"
"| amfi_code | INTEGER | AMFI code (FK) | 119551 |\n"
"| transaction_date | DATE | Date of transaction | 2023-01-15 |\n"
"| transaction_type | TEXT | SIP/Lumpsum/Redemption | SIP |\n"
"| amount | REAL | Transaction amount Rs. | 5000 |\n"
"| units | REAL | Units purchased | 91.82 |\n"
"| nav_at_txn | REAL | NAV at transaction | 54.45 |\n"
"| investor_id | TEXT | Investor identifier | INV001 |\n"
"| state | TEXT | Investor state | Maharashtra |\n"
"| kyc_status | TEXT | KYC verification | VERIFIED |\n"
"\n"
"---\n"
"\n"
"## 4. fact_performance\n"
"**Source**: 07_scheme_performance.csv\n"
"\n"
"| Column | Data Type | Description | Example |\n"
"|--------|-----------|-------------|---------|\n"
"| amfi_code | INTEGER | AMFI code (FK) | 119551 |\n"
"| return_1m | REAL | 1 month return % | 2.5 |\n"
"| return_3m | REAL | 3 month return % | 7.8 |\n"
"| return_6m | REAL | 6 month return % | 12.3 |\n"
"| return_1y | REAL | 1 year return % | 24.5 |\n"
"| return_3y | REAL | 3 year CAGR % | 18.2 |\n"
"| return_5y | REAL | 5 year CAGR % | 15.6 |\n"
"| expense_ratio | REAL | Expense ratio | 0.66 |\n"
"| has_anomaly | INTEGER | Anomaly flag | 0 |\n"
"\n"
"---\n"
"\n"
"## 5. fact_aum\n"
"**Source**: 03_aum_by_fund_house.csv\n"
"\n"
"| Column | Data Type | Description | Example |\n"
"|--------|-----------|-------------|---------|\n"
"| amfi_code | INTEGER | AMFI code (FK) | 119551 |\n"
"| fund_house | TEXT | AMC name | SBI Mutual Fund |\n"
"| month | DATE | Month of AUM | 2024-01-01 |\n"
"| aum_crores | REAL | AUM in Crores | 45231.5 |\n"
"\n"
"---\n"
"\n"
"## Business Definitions\n"
"\n"
"| Term | Definition |\n"
"|------|------------|\n"
"| NAV | Net Asset Value — price per unit |\n"
"| AUM | Assets Under Management |\n"
"| SIP | Systematic Investment Plan |\n"
"| AMFI | Association of Mutual Funds in India |\n"
"| SEBI | Securities & Exchange Board of India |\n"
"| CAGR | Compound Annual Growth Rate |\n"
"| Expense Ratio | Annual fee as % of AUM |\n"
"| Exit Load | Fee on early redemption |\n"
"| KYC | Know Your Customer |\n"
"| Direct Plan | No distributor commission |\n"
"| Regular Plan | With distributor commission |\n"
"\n"
"---\n"
"\n"
"## Data Quality Notes\n"
"\n"
"| Dataset | Status |\n"
"|---------|--------|\n"
"| fund_master | Cleaned |\n"
"| nav_history | Cleaned + Forward-filled |\n"
"| investor_transactions | Cleaned + Standardized |\n"
"| scheme_performance | Cleaned + Anomalies flagged |\n"
"| aum_by_fund_house | Cleaned |\n"
"\n"
"---\n"
"*Prepared by: MERLIN J | Teyzix Core Internship June 2026*\n"
)

with open('reports/data_dictionary.md', 'w',
          encoding='utf-8') as f:
    f.write(data_dict)



In [8]:
import os

files = {
    'data/processed/08_investor_transactions_clean.csv': 'Transactions Cleaned',
    'reports/data_dictionary.md'                       : 'Data Dictionary',
    'bluestock_mf.db'                                  : 'SQLite Database',
    'sql/schema.sql'                                   : 'Schema SQL',
    'sql/queries.sql'                                  : 'Queries SQL',
    'reports/data_quality_summary.txt'                 : 'Data Quality Summary',
}

print("=" * 55)
print("   MISSING FILES CHECK")
print("=" * 55)

for path, name in files.items():
    if os.path.exists(path):
        size = os.path.getsize(path) / 1024
        print(f"   {name:<35} ({size:.1f} KB)")
    else:
        print(f"   {name:<35} MISSING!")



   MISSING FILES CHECK
   Transactions Cleaned                MISSING!
   Data Dictionary                     (3.9 KB)
   SQLite Database                     (5624.0 KB)
   Schema SQL                          (2.3 KB)
   Queries SQL                         (2.8 KB)
   Data Quality Summary                (0.8 KB)


In [9]:
import pandas as pd

df = pd.read_csv('08_investor_transactions.csv')

print("Columns:")
for col in df.columns:
    print(f"  → {col}")

print(f"\nShape: {df.shape}")
print(f"\nFirst 3 rows:")
print(df.head(3).to_string())

print(f"\nData Types:")
print(df.dtypes.to_string())

print(f"\nUnique values in each column:")
for col in df.columns:
    unique = df[col].nunique()
    sample = df[col].unique()[:5].tolist()
    print(f"  {col}: {unique} unique → {sample}")

Columns:
  → investor_id
  → transaction_date
  → amfi_code
  → transaction_type
  → amount_inr
  → state
  → city
  → city_tier
  → age_group
  → gender
  → annual_income_lakh
  → payment_mode
  → kyc_status

Shape: (32778, 13)

First 3 rows:
  investor_id transaction_date  amfi_code transaction_type  amount_inr      state       city city_tier age_group  gender  annual_income_lakh payment_mode kyc_status
0   INV003054       2024-01-01     119092              SIP        1834  Telangana  Hyderabad       T30       56+  Female                77.1          UPI   Verified
1   INV002952       2024-01-01     148567       Redemption      392882     Punjab   Amritsar       B30     18-25    Male                 7.1       Cheque   Verified
2   INV003420       2024-01-01     118636              SIP         912    Haryana  Faridabad       B30     36-45    Male                47.2      Mandate   Verified

Data Types:
investor_id            object
transaction_date       object
amfi_code              

In [10]:
import pandas as pd
import os

df_inv = pd.read_csv('08_investor_transactions.csv')

print("=" * 55)
print("   CLEANING INVESTOR TRANSACTIONS")
print("=" * 55)
print(f"  Before : {df_inv.shape}")

# 1. Parse date
df_inv['transaction_date'] = pd.to_datetime(
    df_inv['transaction_date'])
print("  transaction_date parsed to datetime")

# 2. Transaction type already clean!
print(f"\n  Transaction Types:")
print(df_inv['transaction_type'].value_counts().to_string())
print("   transaction_type already standardized!")

# 3. Validate amount > 0
before = len(df_inv)
df_inv = df_inv[df_inv['amount_inr'] > 0]
print(f"\n   Invalid amounts removed: {before - len(df_inv)}")

# 4. Standardize KYC status
print(f"\n  KYC Before:")
print(df_inv['kyc_status'].value_counts().to_string())
df_inv['kyc_status'] = df_inv['kyc_status'].str.strip().str.upper()
print(f"  KYC After:")
print(df_inv['kyc_status'].value_counts().to_string())
print("   KYC status standardized")

# 5. Standardize city_tier
print(f"\n  City Tier:")
print(df_inv['city_tier'].value_counts().to_string())

# 6. Remove duplicates
before = len(df_inv)
df_inv.drop_duplicates(inplace=True)
print(f"\n   Duplicates removed: {before - len(df_inv)}")

# 7. Check missing values
print(f"\n  Missing values:")
print(df_inv.isnull().sum().to_string())

print(f"\n  After  : {df_inv.shape}")

# Save
os.makedirs('data/processed', exist_ok=True)
df_inv.to_csv(
    'data/processed/08_investor_transactions_clean.csv',
    index=False)

size = os.path.getsize(
    'data/processed/08_investor_transactions_clean.csv')/1024

print(f"   Records : {len(df_inv):,}")
print(f"   Size    : {size:.1f} KB")
print(f"   Path    : data/processed/08_investor_transactions_clean.csv")

   CLEANING INVESTOR TRANSACTIONS
  Before : (32778, 13)
  transaction_date parsed to datetime

  Transaction Types:
transaction_type
SIP           19716
Lumpsum        8095
Redemption     4967
   transaction_type already standardized!

   Invalid amounts removed: 0

  KYC Before:
kyc_status
Verified    30146
Pending      2632
  KYC After:
kyc_status
VERIFIED    30146
PENDING      2632
   KYC status standardized

  City Tier:
city_tier
T30    21719
B30    11059

   Duplicates removed: 0

  Missing values:
investor_id           0
transaction_date      0
amfi_code             0
transaction_type      0
amount_inr            0
state                 0
city                  0
city_tier             0
age_group             0
gender                0
annual_income_lakh    0
payment_mode          0
kyc_status            0

  After  : (32778, 13)
   Records : 32,778
   Size    : 3086.6 KB
   Path    : data/processed/08_investor_transactions_clean.csv


In [12]:
import os

files = {
    'data/processed/08_investor_transactions_clean.csv': 'Transactions Cleaned',
    'reports/data_dictionary.md'                       : 'Data Dictionary',
    'bluestock_mf.db'                                  : 'SQLite Database',
    'sql/schema.sql'                                   : 'Schema SQL',
    'sql/queries.sql'                                  : 'Queries SQL',
    'reports/data_quality_summary.txt'                 : 'Data Quality Summary',
}

print("=" * 55)
print("   FILES CHECK")
print("=" * 55)

all_ready = True
for path, name in files.items():
    if os.path.exists(path):
        size = os.path.getsize(path) / 1024
        print(f"   {name:<35} ({size:.1f} KB)")
    else:
        print(f"   {name:<35} MISSING!")
        all_ready = False

print(f"\n{'='*55}")
if all_ready:
    print("   ALL FILES READY ")
else:
    print("   Some files still missing")

   FILES CHECK
   Transactions Cleaned                (3086.6 KB)
   Data Dictionary                     (3.9 KB)
   SQLite Database                     (5624.0 KB)
   Schema SQL                          (2.3 KB)
   Queries SQL                         (2.8 KB)
   Data Quality Summary                (0.8 KB)

   ALL FILES READY 


In [9]:
import os
import sqlite3

print("DAY 1 FILES:")
day1 = [
    'data_ingestion.py',
    'live_nav_fetch.py',
    'requirements.txt',
    'data/raw',
    'data/processed',
    'sql',
    'reports',
    'data/raw/hdfc_top100_nav.csv',
    'data/raw/all_5_schemes_nav.csv',
]
for f in day1:
    status = "yes" if os.path.exists(f) else "no"
    print(f"  {status} {f}")

print("\nDAY 2 FILES:")
day2 = [
    'data/processed/01_fund_master_clean.csv',
    'data/processed/02_nav_history_clean.csv',
    'data/processed/03_aum_by_fund_house_clean.csv',
    'data/processed/04_monthly_sip_inflows_clean.csv',
    'data/processed/05_category_inflows_clean.csv',
    'data/processed/06_industry_folio_count_clean.csv',
    'data/processed/07_scheme_performance_clean.csv',
    'data/processed/08_investor_transactions_clean.csv',
    'data/processed/09_portfolio_holdings_clean.csv',
    'data/processed/10_benchmark_indices_clean.csv',
    'bluestock_mf.db',
    'sql/schema.sql',
    'sql/queries.sql',
    'reports/data_dictionary.md',
    'reports/data_quality_summary.txt',
]
for f in day2:
    status = "yes" if os.path.exists(f) else "no"
    print(f"  {status} {f}")

print("\nSQLITE TABLES:")
if os.path.exists('bluestock_mf.db'):
    conn = sqlite3.connect('bluestock_mf.db')
    cur  = conn.cursor()
    cur.execute(
        "SELECT name FROM sqlite_master WHERE type='table'")
    for t in cur.fetchall():
        cur.execute(f"SELECT COUNT(*) FROM {t[0]}")
        print(f"   {t[0]:<30} {cur.fetchone()[0]:,} rows")
    conn.close()

DAY 1 FILES:
  yes data_ingestion.py
  yes live_nav_fetch.py
  yes requirements.txt
  yes data/raw
  yes data/processed
  yes sql
  yes reports
  yes data/raw/hdfc_top100_nav.csv
  yes data/raw/all_5_schemes_nav.csv

DAY 2 FILES:
  yes data/processed/01_fund_master_clean.csv
  yes data/processed/02_nav_history_clean.csv
  yes data/processed/03_aum_by_fund_house_clean.csv
  yes data/processed/04_monthly_sip_inflows_clean.csv
  yes data/processed/05_category_inflows_clean.csv
  yes data/processed/06_industry_folio_count_clean.csv
  yes data/processed/07_scheme_performance_clean.csv
  yes data/processed/08_investor_transactions_clean.csv
  yes data/processed/09_portfolio_holdings_clean.csv
  yes data/processed/10_benchmark_indices_clean.csv
  yes bluestock_mf.db
  yes sql/schema.sql
  yes sql/queries.sql
  yes reports/data_dictionary.md
  yes reports/data_quality_summary.txt

SQLITE TABLES:
   dim_fund                       40 rows
   fact_nav                       64,320 rows
   fact_tra

In [8]:
!del "C:/Users/MERLIN J/.git/index.lock"

The system cannot find the path specified.


In [10]:
# Run this to see what's in your user folder
import os
print(os.getcwd())
!dir "C:/Users/MERLIN J"

C:\Users\MERLIN J
 Volume in drive C is Windows
 Volume Serial Number is CA9B-F22D

 Directory of C:\Users\MERLIN J

29-06-2026  11:05    <DIR>          .
13-06-2025  09:09    <DIR>          ..
11-03-2026  21:32    <DIR>          .anaconda
02-04-2026  09:52    <DIR>          .cache
13-04-2026  08:16    <DIR>          .codex
29-06-2026  11:06    <DIR>          .conda
11-03-2026  21:43    <DIR>          .continuum
25-06-2026  11:07               127 .gitconfig
13-10-2024  06:44    <DIR>          .idlerc
28-06-2026  18:59    <DIR>          .ipynb_checkpoints
11-03-2026  21:46    <DIR>          .ipython
11-03-2026  21:46    <DIR>          .jupyter
08-04-2026  17:36                20 .lesshst
27-04-2026  21:13    <DIR>          .m2
11-03-2026  23:44    <DIR>          .matplotlib
20-10-2024  21:23    <DIR>          .ms-ad
02-04-2026  10:01               115 .python_history
02-04-2026  09:32    <DIR>          .rustup
12-06-2026  14:58    <DIR>          .streamlit
07-04-2026  01:44    <DIR>   

In [ ]:
import os

# Make sure you're in your PROJECT folder, not user folder
%cd "C:/Users/MERLIN J/your-project-folder-name"

# Delete lock file if exists
!del ".git\index.lock" 2>nul

# Remove old remote and re-add correct one
!git remote remove origin
!git remote add origin https://github.com/EVANGELINAMERLIN/Mutual-Fund-Analytics.git

# Add and commit
!git add .
!git commit -m "first commit"
!git branch -M main
!git push -u origin main